## Question 1
Converting PDFs to Markdown

In [27]:
!uv add 'markitdown[pdf]'


Resolved 158 packages in 138ms
Checked 141 packages in 51ms


In [28]:
import pandas as pd

books_csv_url = 'https://raw.githubusercontent.com/alexeygrigorev/ai-engineering-buildcamp-code/main/01-foundation/homework/books.csv'
books_df = pd.read_csv(books_csv_url)
books_df


,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,https://greenteapress.com/thinkpython2/thinkpy...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,https://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,https://greenteapress.com/complexity2/thinkcom...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,https://greenteapress.com/thinkjava7/thinkjava...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...
5,Think OS,https://greenteapress.com/wp/think-os/,https://greenteapress.com/thinkos/thinkos.pdf
6,Think C++,https://greenteapress.com/wp/think-c/,https://raw.githubusercontent.com/tscheffl/Thi...


In [29]:
import requests
from pathlib import Path

books_dir = Path('books')
books_dir.mkdir(exist_ok=True)

for _, row in books_df.iterrows():
    pdf_response = requests.get(row['pdf_url'])
    filename = books_dir / f"{row['title']}.pdf"
    filename.write_bytes(pdf_response.content)
    print(f"Saved {filename}")


Saved books/Think Python 2e.pdf
Saved books/Think DSP.pdf
Saved books/Think Complexity 2e.pdf
Saved books/Think Java 2e.pdf
Saved books/Physical Modeling in MATLAB.pdf
Saved books/Think OS.pdf
Saved books/Think C++.pdf


In [30]:
from markitdown import MarkItDown

books_text_dir = Path('books_text')
books_text_dir.mkdir(exist_ok=True)

md_converter = MarkItDown()

for pdf_path in books_dir.glob('*.pdf'):
    result = md_converter.convert(str(pdf_path))
    text_path = books_text_dir / f"{pdf_path.stem}.md"
    text_path.write_text(result.text_content)
    print(f"Converted {pdf_path.name} -> {text_path.name}")

Converted Think OS.pdf -> Think OS.md
Converted Think Java 2e.pdf -> Think Java 2e.md
Converted Physical Modeling in MATLAB.pdf -> Physical Modeling in MATLAB.md
Converted Think Python 2e.pdf -> Think Python 2e.md
Converted Think DSP.pdf -> Think DSP.md
Converted Think Complexity 2e.pdf -> Think Complexity 2e.md
Converted Think C++.pdf -> Think C++.md


In [31]:
think_python_path = books_text_dir / 'Think Python 2e.md'
line_count = len(think_python_path.read_text().splitlines())
line_count

9627

## Question 2
Chunking for RAG

In [32]:
# Read each markdown file, clean lines, build the doc dicts

from pathlib import Path

books_text_dir = Path('books_text')

book_docs = []
for md_path in books_text_dir.glob('*.md'):
    lines = md_path.read_text().split('\n')
    non_empty_lines = [line for line in lines if line.strip()]
    book_docs.append({
        'source': md_path.name,
        'content': non_empty_lines
    })

len(book_docs)

7

In [33]:
!uv add gitsource

Resolved 158 packages in 22ms
Checked 141 packages in 33ms


In [34]:
from gitsource import chunk_documents

book_chunks = chunk_documents(book_docs, size=100, step=50)
len(book_chunks)


995

In [35]:
think_python_chunks = [c for c in book_chunks if c['source'] == 'Think Python 2e.md']
len(think_python_chunks)

187

## Question 3
Index with minisearch

In [36]:
# Turn each chunk's line-list content back into a string

documents = []
for chunk in book_chunks:
    doc = chunk.copy()
    doc['content'] = '\n'.join(chunk['content'])
    documents.append(doc)

In [37]:
!uv add minsearch

Resolved 158 packages in 5ms
Checked 141 packages in 4ms


In [38]:
from minsearch import Index

index = Index(text_fields=["content"])
index.fit(documents)

In [39]:
len(documents)


995

In [40]:
results = index.search("python function definition", num_results=5)
results

[{'start': 1800,
  'content': "| ------ | ------------ | --- | --- |\n| total  | = total +    | a;  |     |\nend\nans = total\nIfyouwereusinganyofthosevariablenamesbeforecallingthisscript, youmightbesurprised\nto find, after running the script, that their values had changed. If you have two scripts that\n48 Functions\nuse the same variable names, you might find that they work separately and then break when\nyou try to combine them. This kind of interaction is called a collision.\nname\nAs the number of scripts you write increases, and they get longer and more complex, name\ncollisions become more of a problem. Avoiding this problem is one of several motivations for\nfunctions.\n| 5.2 | Defining | Functions |     |     |\n| --- | -------- | --------- | --- | --- |\nA function is like a script, except that each function has its own workspace, so any variables\ndefined inside a function only exist while the function is running and don’t interfere with\nvariables in other workspaces, even 

# Question 4
Do RAG

In [41]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [24]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context
    ).strip()
    return prompt

def search(question):
    return index.search(question, num_results=5)

def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer


In [42]:
rag("python function definition")


"To define a function in Python, you start with the `def` keyword, followed by the function name and parentheses that may include parameters. The body of the function follows, typically indented. Here's a basic example:\n\n```python\ndef myfunc(x):\n    s = sin(x)\n    c = cos(x)\n    res = abs(s) + abs(c)\n    return res\n```\n\n### Key Elements of a Function Definition:\n\n1. **Function Name**: Choose a descriptive name, like `myfunc`.\n2. **Parameters**: Specify inputs in parentheses (e.g., `x`).\n3. **Function Body**: Contains the code that executes when the function is called.\n4. **Return Statement**: Use `return` to send a value back to the caller.\n\n### Example Usage:\n\nTo call the function and store the result:\n\n```python\nresult = myfunc(1)\nprint(result)\n```\n\nThis structure allows for better organization and avoidance of variable name collisions across different parts of your code."

In [43]:
def llm(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return {
        'answer': response.output_text,
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens
    }

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm(prompt, instructions)
    return result


In [44]:
result = rag("python function definition")
result['input_tokens']


7925

# Question 6
Structured vs unstructured output

In [45]:
from pydantic import BaseModel, Field
from typing import Literal

class RAGResponse(BaseModel):
    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions")


In [46]:
def llm_structured(user_prompt, instructions, model='gpt-4o-mini'):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.responses.parse(
        model=model,
        input=messages,
        text_format=RAGResponse
    )

    return {
        'answer': response.output_parsed,
        'input_tokens': response.usage.input_tokens,
        'output_tokens': response.usage.output_tokens
    }

def rag_structured(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    result = llm_structured(prompt, instructions)
    return result


In [47]:
structured_result = rag_structured("python function definition")
structured_result['input_tokens']


8111

In [48]:
structured_result['input_tokens'] - result['input_tokens']


186